In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
paths = get_paths(ROOT)

train_df = pd.read_parquet(paths.data_processed / "train.parquet")
val_df   = pd.read_parquet(paths.data_processed / "val.parquet")
test_df  = pd.read_parquet(paths.data_processed / "test.parquet")

from subsample import SubsampleConfig, subsample_splits_to_total, save_subsampled_splits

cfg_sub = SubsampleConfig(total_reviews=120_000, train_frac=0.8, val_frac=0.1, test_frac=0.1, seed=42)
train_80k, val_80k, test_80k = subsample_splits_to_total(train_df, val_df, test_df, cfg_sub)

print(len(train_80k), len(val_80k), len(test_80k), "tot:", len(train_80k)+len(val_80k)+len(test_80k))
print("pos rate:", train_80k["label"].mean(), val_80k["label"].mean(), test_80k["label"].mean())

p_train, p_val, p_test = save_subsampled_splits(train_80k, val_80k, test_80k, paths.data_processed, prefix="80k")
p_train, p_val, p_test


96000 12000 12000 tot: 120000
pos rate: 0.26296875 0.263 0.263


(WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/train_80k.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/val_80k.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/test_80k.parquet'))

In [2]:
from windowing import WindowingConfig, chunk_parquet_streaming

TOKENIZER_NAME = "bert-large-uncased"  # così poi sei allineato col teacher
cfg_win = WindowingConfig(tokenizer_name=TOKENIZER_NAME, max_length=256, stride=128, review_batch_size=512)

out_train_chunks = paths.data_processed / "train_80k_chunks.parquet"
out_val_chunks   = paths.data_processed / "val_80k_chunks.parquet"
out_test_chunks  = paths.data_processed / "test_80k_chunks.parquet"

chunk_parquet_streaming(p_train, out_train_chunks, cfg_win)
chunk_parquet_streaming(p_val, out_val_chunks, cfg_win)
chunk_parquet_streaming(p_test, out_test_chunks, cfg_win)

list(paths.data_processed.iterdir())


c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/reviews.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/test.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/test_80k.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/test_80k_chunks.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/train.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/train_80k.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/train_80k_chunks.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/val.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/val_80k.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/val_80k_chunks.parquet')]

In [3]:
train_chunks = pd.read_parquet(out_train_chunks)
chunk_counts = (
    train_chunks.groupby("review_id")["chunk_index"]
    .max()
    .add(1)
    .rename("n_chunks")
    .reset_index()
)
chunk_counts.head()
chunk_counts.to_parquet(paths.data_processed / "train_120k_chunk_counts.parquet", index=False)

print("train chunks:", len(train_chunks))
print("pos rate chunks:", train_chunks["label"].mean())

# chunk per review
chunks_per_review = train_chunks.groupby("review_id")["chunk_index"].max() + 1
print("chunks/review mean:", chunks_per_review.mean(), "median:", chunks_per_review.median(), "max:", chunks_per_review.max())


train chunks: 212943
pos rate chunks: 0.3414434848762345
chunks/review mean: 2.21815625 median: 1.0 max: 22
